# FMP Currencies

Read-first examples for FMP-backed FX OHLCV data.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openbb import obb

for env_dir in [Path.cwd(), *Path.cwd().parents]:
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break

from quant_warehouse.ingest.credentials import configure_openbb_credentials
from quant_warehouse.ingest.openbb_fetch import SECTION_ROUTES
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.sections import (
    CRYPTO_PRICES_LIBRARY,
    CRYPTO_PRICES_SECTION,
    CURRENCY_PRICES_LIBRARY,
    CURRENCY_PRICES_SECTION,
    DEFAULT_CRYPTO_SYMBOLS,
    DEFAULT_CURRENCY_SYMBOLS,
    DEFAULT_ECONOMIC_SERIES,
    DEFAULT_INDEX_SYMBOLS,
    EQUITY_CALENDAR_SECTIONS,
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    ETF_PRICES_SECTION,
    FUND_PRICES_LIBRARY,
    FUND_PRICES_SECTION,
    INDEX_PRICES_LIBRARY,
    INDEX_PRICES_SECTION,
    MACRO_CALENDAR_LIBRARY,
    MACRO_CALENDAR_SECTION,
    MACRO_CALENDAR_SYMBOL,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_ECONOMIC_SECTION,
    MACRO_RISK_PREMIUM_LIBRARY,
    MACRO_RISK_PREMIUM_SECTION,
    MACRO_TREASURY_LIBRARY,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_LIBRARY,
    MACRO_YIELD_CURVE_SECTION,
    RISK_PREMIUM_SYMBOL,
    TREASURY_BUNDLE_SYMBOL,
    YIELD_CURVE_BUNDLE_SYMBOL,
    fundamental_library,
)
from quant_warehouse.warehouse.storage import provider_library

configure_openbb_credentials()

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()
PROVIDER = "fmp"
RUN_REFRESH = False

CURRENCY_SYMBOL = "EURUSD"

In [2]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def route_table(section_to_base_library: dict[str, str]) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "section": section,
                "openbb_route": SECTION_ROUTES.get(section),
                "provider_library": provider_library(base_library, PROVIDER),
            }
            for section, base_library in section_to_base_library.items()
        ]
    )


def run_openbb_route(label: str, fn):
    try:
        result = fn()
        frame = result.to_df()
        return {
            "label": label,
            "provider": getattr(result, "provider", None),
            "rows": 0 if frame is None else len(frame),
            "columns": [] if frame is None else list(frame.columns),
            "error": None,
        }
    except Exception as exc:
        return {
            "label": label,
            "provider": PROVIDER,
            "rows": None,
            "columns": [],
            "error": str(exc),
        }

In [3]:
currency_routes = {CURRENCY_PRICES_SECTION: CURRENCY_PRICES_LIBRARY}
route_table(currency_routes)

,section,openbb_route,provider_library
0,currency_prices,currency.price.historical,fmp_currency_prices


In [4]:
pd.Series(DEFAULT_CURRENCY_SYMBOLS, name="default_currency_symbols")

0    EURUSD
1    GBPUSD
2    USDJPY
3    AUDUSD
4    USDCAD
5    USDCHF
6    NZDUSD
Name: default_currency_symbols, dtype: object

In [5]:
if RUN_REFRESH:
    warehouse.market_prices.refresh(CURRENCY_SYMBOL, section=CURRENCY_PRICES_SECTION, provider=PROVIDER)

state_table(CURRENCY_SYMBOL, [CURRENCY_PRICES_SECTION])

,symbol,section,provider,min_date,max_date,row_count,column_count,columns_present,last_fetched_at
0,EURUSD,currency_prices,fmp,2007-06-11,2026-06-19,5000,8,"(close, fmp__change, fmp__change_percent, fmp_...",2026-06-20T16:09:56.352126+00:00


In [6]:
preview(warehouse.market_prices.read(CURRENCY_SYMBOL, section=CURRENCY_PRICES_SECTION, provider=PROVIDER))

,open,high,low,close,volume,fmp__vwap,fmp__change,fmp__change_percent
date,,,,,,,,
2026-06-15,1.15752,1.16221,1.15726,1.15894,205438.0,1.15898,0.00142,0.001227
2026-06-16,1.15939,1.16198,1.15748,1.16085,92692.0,1.15993,0.00146,0.001259
2026-06-17,1.16098,1.16168,1.14779,1.15020,158703.0,1.15516,-0.01078,-0.009285
2026-06-18,1.15013,1.15283,1.14513,1.14579,149113.0,1.14847,-0.00434,-0.003774
2026-06-19,1.14630,1.14807,1.14180,1.14712,97014.0,1.14582,0.00082,0.000715


<!-- quant-trader-eda -->
## Quant Trader EDA

FX checks: return/risk profile, rolling carry-free trend proxies, drawdown, and recent range behavior.

In [7]:
def price_eda(frame: pd.DataFrame, annualization: int = 252) -> tuple[pd.DataFrame, pd.DataFrame]:
    if frame is None or frame.empty or "close" not in frame.columns:
        return pd.DataFrame(), pd.DataFrame()
    close = pd.to_numeric(frame["close"], errors="coerce").dropna()
    returns = close.pct_change().dropna()
    if returns.empty:
        return pd.DataFrame(), pd.DataFrame()
    equity_curve = (1 + returns).cumprod()
    drawdown = equity_curve / equity_curve.cummax() - 1
    summary = pd.DataFrame(
        {
            "observations": [int(len(returns))],
            "start": [close.index.min()],
            "end": [close.index.max()],
            "total_return": [close.iloc[-1] / close.iloc[0] - 1],
            "annualized_return": [(1 + returns.mean()) ** annualization - 1],
            "annualized_volatility": [returns.std() * annualization ** 0.5],
            "sharpe_0_rf": [returns.mean() / returns.std() * annualization ** 0.5 if returns.std() else None],
            "max_drawdown": [drawdown.min()],
            "best_day": [returns.max()],
            "worst_day": [returns.min()],
        }
    )
    diagnostics = pd.DataFrame(
        {
            "close": close,
            "return": returns.reindex(close.index),
            "rolling_21d_vol": returns.rolling(21).std() * annualization ** 0.5,
            "rolling_63d_vol": returns.rolling(63).std() * annualization ** 0.5,
            "drawdown": drawdown.reindex(close.index),
        }
    )
    return summary, diagnostics

In [8]:
currency_prices = warehouse.market_prices.read(CURRENCY_SYMBOL, section=CURRENCY_PRICES_SECTION, provider=PROVIDER)
fx_summary, fx_diagnostics = price_eda(currency_prices)
fx_summary

,observations,start,end,total_return,annualized_return,annualized_volatility,sharpe_0_rf,max_drawdown,best_day,worst_day
0,4999,2007-06-11,2026-06-19,-0.141191,-0.003779,0.088189,-0.042935,-0.40012,0.03492,-0.026723


In [9]:
preview(fx_diagnostics[["close", "rolling_21d_vol", "rolling_63d_vol", "drawdown"]], rows=10)

,close,rolling_21d_vol,rolling_63d_vol,drawdown
date,,,,
2026-06-09,1.15440,0.038478,0.049425,-0.278103
2026-06-10,1.15357,0.038500,0.049324,-0.278622
2026-06-11,1.15792,0.039506,0.049564,-0.275902
2026-06-12,1.15669,0.036945,0.047007,-0.276671
2026-06-14,1.16027,0.038169,0.046964,-0.274432
2026-06-15,1.15894,0.038292,0.046170,-0.275264
2026-06-16,1.16085,0.038498,0.046157,-0.274069
2026-06-17,1.15020,0.048459,0.049718,-0.280729
2026-06-18,1.14579,0.049578,0.049944,-0.283487


In [10]:
if currency_prices.empty or "close" not in currency_prices.columns:
    pd.DataFrame()
else:
    close = pd.to_numeric(currency_prices["close"], errors="coerce").dropna()
    fx_features = pd.DataFrame({
        "close": close,
        "rolling_20d_high": close.rolling(20).max(),
        "rolling_20d_low": close.rolling(20).min(),
        "distance_to_60d_sma": close / close.rolling(60).mean() - 1,
    })
    preview(fx_features, rows=10)

<!-- quant-trader-signal-eda -->
## Signal Research for P&L

Without carry data, this notebook focuses on price-only FX signals: trend, range breakout, realized volatility, and drawdown control.

In [11]:
def signal_backtest(close: pd.Series, signal: pd.Series, annualization: int = 252) -> pd.DataFrame:
    close = pd.to_numeric(close, errors="coerce").dropna()
    returns = close.pct_change().dropna()
    aligned_signal = signal.reindex(returns.index).shift(1).fillna(0).clip(-1, 1)
    strategy_returns = aligned_signal * returns
    if strategy_returns.empty:
        return pd.DataFrame()
    curve = (1 + strategy_returns).cumprod()
    buy_hold = (1 + returns).cumprod()
    drawdown = curve / curve.cummax() - 1
    turnover = aligned_signal.diff().abs().fillna(aligned_signal.abs())
    return pd.DataFrame({
        "strategy_total_return": [curve.iloc[-1] - 1],
        "buy_hold_total_return": [buy_hold.iloc[-1] - 1],
        "strategy_annualized_return": [(1 + strategy_returns.mean()) ** annualization - 1],
        "strategy_annualized_volatility": [strategy_returns.std() * annualization ** 0.5],
        "strategy_sharpe_0_rf": [strategy_returns.mean() / strategy_returns.std() * annualization ** 0.5 if strategy_returns.std() else None],
        "strategy_max_drawdown": [drawdown.min()],
        "hit_rate": [(strategy_returns > 0).mean()],
        "average_exposure": [aligned_signal.abs().mean()],
        "average_daily_turnover": [turnover.mean()],
        "latest_signal": [aligned_signal.iloc[-1]],
    })

In [12]:
currency_prices = warehouse.market_prices.read(CURRENCY_SYMBOL, section=CURRENCY_PRICES_SECTION, provider=PROVIDER)
if currency_prices.empty or "close" not in currency_prices.columns:
    pd.DataFrame()
else:
    close = currency_prices["close"]
    trend_signal = (close > close.rolling(100).mean()).astype(float) * 2 - 1
    signal_backtest(close, trend_signal)

In [13]:
if currency_prices.empty or "close" not in currency_prices.columns:
    pd.DataFrame()
else:
    close = currency_prices["close"]
    returns = close.pct_change()
    breakout = pd.DataFrame({
        "close": close,
        "breakout_60d_high_distance": close / close.rolling(60).max() - 1,
        "breakout_60d_low_distance": close / close.rolling(60).min() - 1,
        "realized_vol_21d": returns.rolling(21).std() * 252 ** 0.5,
        "zscore_60d": (close - close.rolling(60).mean()) / close.rolling(60).std(),
    })
    preview(breakout, rows=15)

<!-- output-analysis -->
## Analysis Notes From This Run

For `EURUSD`, the stored FMP series is slightly negative over the sample, with about 8.8% annualized volatility and a 40.0% max drawdown. The latest close is below the 200-day average by about 1.7%, so the current price-only state is mildly bearish but not an extreme dislocation.

Because this notebook does not include interest-rate carry, the price-only FX signals should be treated as incomplete. The useful trading work here is trend/range diagnostics and volatility sizing; a production FX model should add carry, rate differentials, and transaction cost assumptions.